# Unsloth Qwen3 (4B) Finetuning on `entry2exit_df`

Adapted from the Unsloth Qwen3 Instruct notebook:
- https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)-Instruct.ipynb

This notebook fine-tunes a Qwen3 instruct model for:
- input: `input_text`
- output: `target_text`

Expected CSV from your other notebook:
- `entry2exit.csv` with columns `input_text`, `target_text` (optional `entry_date` for time-aware split)

In [1]:
# Install (Colab / Linux CUDA recommended for Unsloth)
# If needed once:
!pip install unsloth transformers==4.56.2 trl==0.22.2 datasets accelerate bitsandbytes sentencepiece protobuf

In [2]:
import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('Warning: Unsloth training is intended for CUDA GPUs. CPU/MPS is likely too slow or unsupported.')

Skipping import of cpp extensions due to incompatible torch version 2.9.1 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0219 14:39:38.347000 46293 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
/var/folders/v7/4r0w4bz13n53t7r_hr77cp1w0000gn/T/ipykernel_46293/2997936066.py:10: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


NotImplementedError: Unsloth currently only works on NVIDIA, AMD and Intel GPUs.

In [ ]:
# 1) Load CSV generated by the other notebook + clean dataframe
from pathlib import Path

ENTRY2EXIT_CSV_PATH = Path("./entry2exit.csv")
if not ENTRY2EXIT_CSV_PATH.exists():
    raise RuntimeError(
        f"CSV not found at {ENTRY2EXIT_CSV_PATH.resolve()}. "
        "Generate it in notebook_transformer_ml_standalone.ipynb via "
        "entry2exit_df.to_csv('entry2exit.csv', index=False)."
    )

entry2exit_df = pd.read_csv(ENTRY2EXIT_CSV_PATH)
required_cols = {"input_text", "target_text"}
if not required_cols.issubset(entry2exit_df.columns):
    raise RuntimeError(f"CSV must contain columns {required_cols}. Found: {list(entry2exit_df.columns)}")

seq2seq_df = entry2exit_df.copy()
seq2seq_df = seq2seq_df.dropna(subset=["input_text", "target_text"]).reset_index(drop=True)
seq2seq_df = seq2seq_df[
    seq2seq_df["input_text"].astype(str).str.len().gt(0)
    & seq2seq_df["target_text"].astype(str).str.len().gt(0)
].reset_index(drop=True)

if "entry_date" in seq2seq_df.columns:
    seq2seq_df["entry_date"] = pd.to_datetime(seq2seq_df["entry_date"], errors="coerce")
    seq2seq_df = seq2seq_df.sort_values("entry_date").reset_index(drop=True)

split_idx = int(len(seq2seq_df) * 0.90)
train_df = seq2seq_df.iloc[:split_idx].copy()
val_df = seq2seq_df.iloc[split_idx:].copy()

print('Loaded CSV:', ENTRY2EXIT_CSV_PATH.resolve())
print('Total rows:', len(seq2seq_df))
print('Train rows:', len(train_df))
print('Val rows:', len(val_df))
display(train_df[["input_text", "target_text"]].head(2))


In [ ]:
# 2) Load model/tokenizer (adapted from Unsloth Qwen3 notebook)
max_seq_length = 1024
load_in_4bit = True

# Common options:
# model_name = "unsloth/Qwen3-4B-Instruct"
# model_name = "unsloth/Qwen3-4B-Instruct-2507"
model_name = "unsloth/Qwen3-4B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=load_in_4bit,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen3-instruct",
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
# 3) Build chat-format dataset

def _to_chat_record(row):
    return {
        "conversations": [
            {"role": "user", "content": str(row["input_text"])},
            {"role": "assistant", "content": str(row["target_text"])},
        ]
    }

train_chat_df = train_df[["input_text", "target_text"]].copy()
val_chat_df = val_df[["input_text", "target_text"]].copy()

train_hf = Dataset.from_pandas(train_chat_df, preserve_index=False)
val_hf = Dataset.from_pandas(val_chat_df, preserve_index=False)

train_hf = train_hf.map(lambda x: _to_chat_record(x), remove_columns=train_hf.column_names)
val_hf = val_hf.map(lambda x: _to_chat_record(x), remove_columns=val_hf.column_names)


def _format_chat(ex):
    text = tokenizer.apply_chat_template(
        ex["conversations"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_hf = train_hf.map(_format_chat)
val_hf = val_hf.map(_format_chat)

print(train_hf[0]["text"][:500])

In [ ]:
# 4) Trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=TrainingArguments(
        output_dir="./artifacts/unsloth_qwen3_entry2exit",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        per_device_eval_batch_size=2,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        num_train_epochs=3,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

# Only train on assistant responses (matches Unsloth notebook pattern)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user
",
    response_part="<|im_start|>assistant
",
)

In [ ]:
# 5) Train
train_result = trainer.train()
print(train_result)

In [ ]:
# 6) Save LoRA adapter + tokenizer
save_dir = "./artifacts/unsloth_qwen3_entry2exit"
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print('Saved adapter/tokenizer to:', save_dir)

In [ ]:
# 7) Inference helper + quick samples
FastLanguageModel.for_inference(model)

def generate_exit_text_qwen(input_text: str, max_new_tokens: int = 160) -> str:
    messages = [{"role": "user", "content": input_text}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            num_beams=1,
            repetition_penalty=1.05,
            no_repeat_ngram_size=3,
            use_cache=True,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded

print("\n--- Validation samples ---")
for i in range(min(3, len(val_df))):
    inp = str(val_df.iloc[i]["input_text"])
    tgt = str(val_df.iloc[i]["target_text"])
    pred = generate_exit_text_qwen(inp)
    print(f"\n[{i}] INPUT:\n{inp[:300]}...")
    print(f"TARGET:\n{tgt[:300]}...")
    print(f"PRED:\n{pred[:300]}...")

## Notes
- This is decoder-only instruction tuning (Qwen3 Instruct), not encoder-decoder seq2seq.
- On Apple Silicon (MPS), use your existing FLAN-T5 path instead; Unsloth is CUDA-first.
- If outputs include prompt text, post-process by extracting only the assistant span after generation.